# 05 — Análisis Exploratorio de Datos (EDA) del Dataset Consolidado

Este notebook genera las estadísticas descriptivas, análisis de balance de clases,
distribución espacial en la arena y las figuras gráficas oficiales para el informe técnico de la **Unidad I**.

Lee directamente `metadata/arena_placements.json` y `metadata/cards_catalog.json` desde Cloudflare R2 sin descargar videos.


In [ ]:
import os, json, time
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import boto3
from botocore.config import Config

WORK_DIR = Path('/kaggle/working') if Path('/kaggle').exists() else Path('./working')
FIG_DIR = WORK_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

for env_candidate in [Path('.env'), Path('../.env')]:
    if env_candidate.exists():
        for line in env_candidate.read_text().splitlines():
            line = line.strip()
            if line and not line.startswith('#') and '=' in line:
                k, v = line.split('=', 1)
                os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))

def get_config(key: str, default: str = None) -> str:
    if os.getenv(key): return os.getenv(key)
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(key)
    except Exception: return default

R2_ENDPOINT = get_config('R2_ENDPOINT')
R2_ACCESS_KEY = get_config('R2_ACCESS_KEY')
R2_SECRET_KEY = get_config('R2_SECRET_KEY')
R2_BUCKET = get_config('R2_BUCKET', 'cr-videos')

s3 = boto3.client(
    's3',
    endpoint_url=R2_ENDPOINT,
    aws_access_key_id=R2_ACCESS_KEY,
    aws_secret_access_key=R2_SECRET_KEY,
    config=Config(signature_version='s3v4')
)
print(f"✔ Conectado a R2 ({R2_BUCKET})")


In [ ]:
# Carga de manifiesto consolidado de placements y catálogo
obj = s3.get_object(Bucket=R2_BUCKET, Key='metadata/arena_placements.json')
data = json.loads(obj['Body'].read().decode('utf-8'))

tot_vids = data.get('total_videos', len(data.get('videos', {})))
tot_plays = data.get('total_play_samples', 0)
tot_waits = data.get('total_wait_samples', 0)
tot_samples = tot_plays + tot_waits

print(f"Dataset Maestro: {tot_vids} videos | {tot_samples} decisiones tácticas totales")
print(f"  • Jugadas de cartas: {tot_plays} ({tot_plays / tot_samples * 100:.1f}%)")
print(f"  • Acciones de esperar: {tot_waits} ({tot_waits / tot_samples * 100:.1f}%)")

card_counts = Counter()
zone_counts = Counter()
spawn_methods = Counter()
elixir_spent_list = []
coords_list = []

for vid, vinfo in data.get('videos', {}).items():
    for m in vinfo.get('matches', []):
        for ev in m.get('events', []):
            card_counts[ev.get('card_name_es', 'Desconocida')] += 1
            zone_counts[ev.get('zone_phrase', 'Zona no asignada')] += 1
            spawn_methods[ev.get('spawn_method', 'unknown')] += 1
            if 'estimated_spend' in ev:
                elixir_spent_list.append(ev['estimated_spend'])
            if 'x_norm' in ev and 'y_norm' in ev:
                coords_list.append((ev['x_norm'], ev['y_norm']))

print(f"✔ Estadísticas extraídas: {len(card_counts)} cartas distintas jugadas, {len(zone_counts)} zonas")


In [ ]:
# FIGURA 1: Balance de Acciones (Jugar vs Esperar) y Métodos de Detección
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.2), dpi=150)

# Gráfico 1: Jugar vs Esperar
labels_act = ['Jugadas Activas\\n(Despliegues)', 'Acción Pasiva\\n(esperar)']
sizes_act = [tot_plays, tot_waits]
colors_act = ['#2563eb', '#10b981']
wedges, texts, autotexts = ax1.pie(
    sizes_act, labels=labels_act, autopct='%1.1f%%', startangle=140,
    colors=colors_act, explode=(0.04, 0.04), textprops={'fontsize': 11, 'fontweight': 'bold'}
)
for at in autotexts: at.set_color('white')
ax1.set_title('Balance del Dataset: Jugadas vs Esperas', fontsize=12, fontweight='bold', pad=12)

# Gráfico 2: Desglose del Detector de Spawn
m_labels = ['Etiqueta Flotante\n(Texto blanco/dorado)', 'Aparición Neta\n(Fallback movimiento)']
m_vals = [spawn_methods['label'], spawn_methods['motion']]
m_colors = ['#059669', '#f59e0b']
bars = ax2.bar(m_labels, m_vals, color=m_colors, width=0.55, edgecolor='#334155', linewidth=1.2)
for bar in bars:
    h = bar.get_height()
    pct = h / sum(m_vals) * 100
    ax2.text(bar.get_x() + bar.get_width()/2, h + 30, f'{h} ({pct:.1f}%)', ha='center', fontsize=11, fontweight='bold')
ax2.set_ylim(0, max(m_vals) * 1.15)
ax2.set_title('Efectividad del Anclaje de Despliegue en Arena', fontsize=12, fontweight='bold', pad=12)
ax2.set_ylabel('Número de Jugadas', fontsize=10)

plt.tight_layout()
f1_path = FIG_DIR / 'eda_balance_y_deteccion.png'
plt.savefig(f1_path, bbox_inches='tight')
plt.show()
print(f"✔ Guardado: {f1_path}")


In [ ]:
# FIGURA 2: Top 15 Cartas más Jugadas en el Dataset
fig, ax = plt.subplots(figsize=(10, 6), dpi=150)
top15 = card_counts.most_common(15)
names = [x[0] for x in reversed(top15)]
counts = [x[1] for x in reversed(top15)]

bars = ax.barh(names, counts, color='#3b82f6', edgecolor='#1e40af', height=0.65)
for bar in bars:
    w = bar.get_width()
    ax.text(w + 3, bar.get_y() + bar.get_height()/2, f'{w}', va='center', fontsize=9.5, fontweight='bold', color='#1e293b')

ax.set_xlim(0, max(counts) * 1.12)
ax.set_title('Distribución de Frecuencia: Top 15 Cartas más Desplegadas', fontsize=13, fontweight='bold', pad=14)
ax.set_xlabel('Frecuencia de Despliegue en Partidas', fontsize=10.5)
plt.tight_layout()
f2_path = FIG_DIR / 'eda_top_cartas.png'
plt.savefig(f2_path, bbox_inches='tight')
plt.show()
print(f"✔ Guardado: {f2_path}")


In [ ]:
# FIGURA 3: Distribución Espacial de Jugadas por Zona Táctica
fig, ax = plt.subplots(figsize=(11, 5.5), dpi=150)
sorted_zones = zone_counts.most_common()
z_names = [z[0].capitalize() for z in reversed(sorted_zones)]
z_vals = [z[1] for z in reversed(sorted_zones)]

colors_zone = ['#0ea5e9' if 'rival' in n.lower() else '#10b981' for n in z_names]
bars = ax.barh(z_names, z_vals, color=colors_zone, edgecolor='#334155', height=0.65)
for bar in bars:
    w = bar.get_width()
    pct = w / tot_plays * 100
    ax.text(w + 6, bar.get_y() + bar.get_height()/2, f'{w} ({pct:.1f}%)', va='center', fontsize=9.5, fontweight='bold')

ax.set_xlim(0, max(z_vals) * 1.18)
ax.set_title('Concentración Espacial de Despliegues por Zona de la Arena', fontsize=13, fontweight='bold', pad=14)
ax.set_xlabel('Total de Jugadas Realizadas', fontsize=10.5)

patch_own = mpatches.Patch(color='#10b981', label='Campo Propio (posiciones.txt)')
patch_rival = mpatches.Patch(color='#0ea5e9', label='Campo Rival (Hechizos y Globales)')
ax.legend(handles=[patch_own, patch_rival], loc='lower right', frameon=True)

plt.tight_layout()
f3_path = FIG_DIR / 'eda_distribucion_zonas.png'
plt.savefig(f3_path, bbox_inches='tight')
plt.show()
print(f"✔ Guardado: {f3_path}")


---
## 4. Comparación Didáctica de Métodos de Localización Espacial

Visualización de los dos métodos implementados en el pipeline de visión de la arena:
1. **Etiqueta Flotante (78.5%):** Detección tipográfica del nombre de la carta y el reloj de despliegue.
2. **Aparición Neta / Modo Fallback (21.5%):** Resta diferencial de intensidades $|I_{post} - I_{pre}|$ aislando el centroide de la nueva entidad colocada sobre el césped.


In [ ]:
# FIGURA 4: Comparación Visual de Métodos de Localización (Etiqueta vs Resta de Intensidades)
import cv2
import numpy as np

EXAMPLE_VID = "soLMMwrsFIo"
video_url = f"https://cdn-cr.eddux.dev/videos/{EXAMPLE_VID}.mp4"
local_video_path = Path(f"/tmp/{EXAMPLE_VID}.mp4")

if not local_video_path.exists():
    import urllib.request
    print(f"Descargando video de ejemplo {EXAMPLE_VID}...")
    req = urllib.request.Request(video_url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as r, open(local_video_path, "wb") as f:
        f.write(r.read())

obj_layout = s3.get_object(Bucket=R2_BUCKET, Key=f"metadata/layouts/{EXAMPLE_VID}.json")
layout = json.loads(obj_layout["Body"].read().decode("utf-8"))
ax, ay, aw, ah = layout["boxes_pixel"]["arena_box"]

cap = cv2.VideoCapture(str(local_video_path))

# 1. Etiqueta Flotante (t = 21.93s, Gigante noble en carril derecho)
cap.set(cv2.CAP_PROP_POS_MSEC, 21930)
_, fr_label = cap.read()
px_g = int(ax + 0.8145 * aw)
py_g = int(ay + 0.4555 * ah)
crop_label = fr_label[max(0, py_g-140):min(fr_label.shape[0], py_g+140), max(0, px_g-160):min(fr_label.shape[1], px_g+160)]
crop_label_rgb = cv2.cvtColor(crop_label, cv2.COLOR_BGR2RGB)
cv2.rectangle(crop_label_rgb, (25, 45), (crop_label_rgb.shape[1]-25, crop_label_rgb.shape[0]-55), (34, 197, 94), 3)
cv2.drawMarker(crop_label_rgb, (crop_label_rgb.shape[1]//2, crop_label_rgb.shape[0]-45), (34, 197, 94), cv2.MARKER_CROSS, 20, 3)

# 2. Modo Fallback (Aparición Neta): Frame previo (10.5s), Frame posterior (13.5s) y Resta
cap.set(cv2.CAP_PROP_POS_MSEC, 10500)
_, fr_pre = cap.read()
cap.set(cv2.CAP_PROP_POS_MSEC, 13500)
_, fr_post = cap.read()

px_c = int(ax + 0.50 * aw)
py_c = int(ay + 0.62 * ah)
h_rad, w_rad = 130, 150

crop_pre = cv2.cvtColor(fr_pre[py_c-h_rad : py_c+h_rad, px_c-w_rad : px_c+w_rad], cv2.COLOR_BGR2RGB)
crop_post = cv2.cvtColor(fr_post[py_c-h_rad : py_c+h_rad, px_c-w_rad : px_c+w_rad], cv2.COLOR_BGR2RGB)

diff_gray = np.abs(crop_post.astype(np.float32) - crop_pre.astype(np.float32)).mean(axis=-1)
diff_norm = (diff_gray / (diff_gray.max() + 1e-6) * 255).astype(np.uint8)
diff_color = cv2.applyColorMap(diff_norm, cv2.COLORMAP_INFERNO)
diff_color_rgb = cv2.cvtColor(diff_color, cv2.COLOR_BGR2RGB)
cv2.circle(diff_color_rgb, (diff_color_rgb.shape[1]//2, diff_color_rgb.shape[0]//2 + 10), 38, (34, 197, 94), 3)

# Generación de la figura de 4 paneles sin solapamiento
fig, axes = plt.subplots(1, 4, figsize=(18, 5.2))
plt.subplots_adjust(top=0.82, wspace=0.18)

axes[0].imshow(crop_label_rgb)
axes[0].set_title("1. Etiqueta Flotante (78.5%)\nTexto de nombre + Reloj blanco", fontsize=11, fontweight="bold", color="#15803d", pad=10)
axes[0].axis("off")

axes[1].imshow(crop_pre)
axes[1].set_title("2. Modo Fallback: Frame Previo\n(t = 10.5s: Centro vacio)", fontsize=11, fontweight="bold", color="#334155", pad=10)
axes[1].axis("off")

axes[2].imshow(crop_post)
axes[2].set_title("3. Modo Fallback: Frame Posterior\n(t = 13.5s: Estructura colocada)", fontsize=11, fontweight="bold", color="#334155", pad=10)
axes[2].axis("off")

axes[3].imshow(diff_color_rgb)
axes[3].set_title("4. Resta |Post - Pre| (Aparicion Neta)\nCentroide detectado", fontsize=11, fontweight="bold", color="#b45309", pad=10)
axes[3].axis("off")

fig.suptitle("Pipeline de Localizacion en Arena: Etiqueta Directa vs. Resta Diferencial (Aparicion Neta)", fontsize=14, fontweight="bold", y=0.96)

fig4_path = FIG_DIR / "ejemplo_label_vs_motion.png"
plt.savefig(fig4_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"✔ Guardada {fig4_path}")
